# KuaiRand 指标体系与业务概览

## 目标

建立短视频推荐指标体系，并比较同期正常推荐与随机曝光的整体表现。


## 1. 业务目标与行为漏斗

业务目标：推荐用户感兴趣的视频，提升深度观看和正向互动，同时控制负反馈。

行为漏斗：

```text
曝光 → 有效播放反馈 → 长播 → 互动 → 负反馈
```


## 2. 指标体系设计

| 层级 | 指标名称 | 计算口径 | 业务含义 |
|---|---|---|---|
| 规模 | 曝光量 | 日志记录数 | 推荐规模 |
| 规模 | 活跃用户数 | `user_id` 去重 | 覆盖用户规模 |
| 规模 | 曝光视频数 | `video_id` 去重 | 覆盖内容规模 |
| 规模 | 人均曝光次数 | 曝光量 / 活跃用户数 | 用户平均推荐量 |
| 消费 | 有效播放反馈率 | `is_click=1` / 曝光量 | 曝光后的有效消费反馈 |
| 消费 | 长播率 | `long_view=1` / 曝光量 | 深度观看比例 |
| 消费 | 平均播放时长 | `play_time_ms` 平均值 | 单次曝光观看深度 |
| 消费 | 中位数播放比例 | `play_time_ms / duration_ms` 中位数 | 典型播放完成程度 |
| 互动 | 点赞率 | `is_like=1` / 曝光量 | 正向反馈 |
| 互动 | 评论率 | `is_comment=1` / 曝光量 | 深度参与 |
| 互动 | 关注率 | `is_follow=1` / 曝光量 | 创作者兴趣 |
| 互动 | 转发率 | `is_forward=1` / 曝光量 | 内容传播 |
| 风险 | 负反馈率 | `is_hate=1` / 曝光量 | 用户反感程度 |
| 数据质量 | 零视频时长占比 | `duration_ms=0` / 曝光量 | 播放比例可用性 |

说明：`is_click` 在不同页面形态下含义不同，本文按“有效播放反馈率”解释。


## 3. 数据读取与清洗口径

只比较同期数据：随机曝光与正常推荐均为 `2022-04-22` 至 `2022-05-08`。

清洗口径：删除完全重复记录；保留零视频时长记录；播放比例仅在 `duration_ms>0` 时计算。


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/raw/KuaiRand-Pure/data"

print("项目目录：", PROJECT_ROOT)
print("数据目录存在：", DATA_DIR.exists())


In [ ]:
log_random = pd.read_csv(
    DATA_DIR / "log_random_4_22_to_5_08_pure.csv"
).drop_duplicates().copy()

log_standard = pd.read_csv(
    DATA_DIR / "log_standard_4_22_to_5_08_pure.csv"
).drop_duplicates().copy()

logs = {
    "随机曝光": log_random,
    "正常推荐": log_standard,
}

for name, log in logs.items():
    print("=" * 60)
    print(name)
    print("记录数：", f"{len(log):,}")
    print("日期范围：", log["date"].min(), "-", log["date"].max())
    print("is_rand 分布：", log["is_rand"].value_counts().to_dict())


In [ ]:
for name, log in logs.items():
    log["duration_missing"] = (log["duration_ms"] == 0).astype("int8")
    log["play_ratio"] = pd.NA

    valid_duration = log["duration_ms"] > 0
    log.loc[valid_duration, "play_ratio"] = (
        log.loc[valid_duration, "play_time_ms"]
        / log.loc[valid_duration, "duration_ms"]
    )

    log["event_hour"] = log["hourmin"] // 100


## 4. 核心指标计算

使用统一函数计算两类曝光机制下的指标，确保口径一致。


In [ ]:
def calculate_metrics(log: pd.DataFrame, name: str) -> dict:
    exposure_count = len(log)
    active_users = log["user_id"].nunique()
    exposed_videos = log["video_id"].nunique()
    valid_duration = log["duration_ms"] > 0

    return {
        "数据集": name,
        "曝光量": exposure_count,
        "活跃用户数": active_users,
        "曝光视频数": exposed_videos,
        "人均曝光次数": exposure_count / active_users,
        "有效播放反馈率": log["is_click"].mean(),
        "长播率": log["long_view"].mean(),
        "平均播放时长_秒": log["play_time_ms"].mean() / 1000,
        "中位数播放比例": log.loc[valid_duration, "play_ratio"].median(),
        "点赞率": log["is_like"].mean(),
        "评论率": log["is_comment"].mean(),
        "关注率": log["is_follow"].mean(),
        "转发率": log["is_forward"].mean(),
        "主页进入率": log["is_profile_enter"].mean(),
        "负反馈率": log["is_hate"].mean(),
        "零视频时长占比": log["duration_missing"].mean(),
    }

metrics = pd.DataFrame([
    calculate_metrics(log_random, "随机曝光"),
    calculate_metrics(log_standard, "正常推荐"),
])

metrics


In [ ]:
percent_columns = [
    "有效播放反馈率",
    "长播率",
    "中位数播放比例",
    "点赞率",
    "评论率",
    "关注率",
    "转发率",
    "主页进入率",
    "负反馈率",
    "零视频时长占比",
]

count_columns = ["曝光量", "活跃用户数", "曝光视频数"]

metrics_display = metrics.copy()

for column in percent_columns:
    metrics_display[column] = metrics_display[column].map(lambda x: f"{x:.2%}")

for column in count_columns:
    metrics_display[column] = metrics_display[column].map(lambda x: f"{int(x):,}")

metrics_display["人均曝光次数"] = metrics_display["人均曝光次数"].map(lambda x: f"{x:.2f}")
metrics_display["平均播放时长_秒"] = metrics_display["平均播放时长_秒"].map(lambda x: f"{x:,.2f}")

metrics_display


## 5. 正常推荐与随机曝光差异

差异定义：

```text
绝对差异 = 正常推荐 - 随机曝光
相对差异 = 绝对差异 / 随机曝光
```

该结果为描述性对比，不直接等同于因果提升。


In [ ]:
comparison = (
    metrics.set_index("数据集")
    .T
    .rename_axis("指标")
    .reset_index()
)

comparison["差异_正常推荐-随机曝光"] = (
    comparison["正常推荐"] - comparison["随机曝光"]
)
comparison["相对差异"] = (
    comparison["差异_正常推荐-随机曝光"]
    / comparison["随机曝光"]
)

comparison


In [ ]:
comparison_display = comparison.copy()

rate_metrics = [
    "有效播放反馈率",
    "长播率",
    "中位数播放比例",
    "点赞率",
    "评论率",
    "关注率",
    "转发率",
    "主页进入率",
    "负反馈率",
    "零视频时长占比",
]
count_metrics = ["曝光量", "活跃用户数", "曝光视频数"]

for idx, row in comparison_display.iterrows():
    metric_name = row["指标"]

    if metric_name in rate_metrics:
        comparison_display.loc[idx, "随机曝光"] = f"{row['随机曝光']:.2%}"
        comparison_display.loc[idx, "正常推荐"] = f"{row['正常推荐']:.2%}"
        comparison_display.loc[idx, "差异_正常推荐-随机曝光"] = f"{row['差异_正常推荐-随机曝光'] * 100:+.2f} 个百分点"
        comparison_display.loc[idx, "相对差异"] = f"{row['相对差异']:.2%}"

    elif metric_name in count_metrics:
        comparison_display.loc[idx, "随机曝光"] = f"{row['随机曝光']:,.0f}"
        comparison_display.loc[idx, "正常推荐"] = f"{row['正常推荐']:,.0f}"
        comparison_display.loc[idx, "差异_正常推荐-随机曝光"] = f"{row['差异_正常推荐-随机曝光']:+,.0f}"
        comparison_display.loc[idx, "相对差异"] = f"{row['相对差异']:.2%}"

    else:
        comparison_display.loc[idx, "随机曝光"] = f"{row['随机曝光']:,.2f}"
        comparison_display.loc[idx, "正常推荐"] = f"{row['正常推荐']:,.2f}"
        comparison_display.loc[idx, "差异_正常推荐-随机曝光"] = f"{row['差异_正常推荐-随机曝光']:+,.2f}"
        comparison_display.loc[idx, "相对差异"] = f"{row['相对差异']:.2%}"

comparison_display


### 初步业务解读

正常推荐在有效播放反馈率、长播率、播放时长和互动指标上整体高于随机曝光，说明算法选片更符合用户兴趣。

该差异包含推荐选择机制影响，暂不能直接解释为严格 A/B 实验中的因果提升。


## 6. 推荐覆盖与集中度

在效果指标之外，继续观察流量是否集中于少数视频。


In [ ]:
def top_video_share(log: pd.DataFrame, top_n: int) -> float:
    video_exposures = log["video_id"].value_counts()
    return video_exposures.head(top_n).sum() / len(log)

coverage = pd.DataFrame([
    {
        "数据集": name,
        "曝光视频数": log["video_id"].nunique(),
        "Top10视频曝光占比": top_video_share(log, 10),
        "Top100视频曝光占比": top_video_share(log, 100),
        "Top1000视频曝光占比": top_video_share(log, 1000),
    }
    for name, log in logs.items()
])

coverage_display = coverage.copy()
for column in ["Top10视频曝光占比", "Top100视频曝光占比", "Top1000视频曝光占比"]:
    coverage_display[column] = coverage_display[column].map(lambda x: f"{x:.2%}")
coverage_display["曝光视频数"] = coverage_display["曝光视频数"].map(lambda x: f"{int(x):,}")

coverage_display


### 覆盖与集中度解读

若正常推荐 Top 视频曝光占比更高，说明推荐分发更集中。后续需结合视频类型、时长和作者维度判断集中原因。


## 7. 本阶段结论

- 正常推荐整体消费和互动表现优于随机曝光。
- 负反馈率作为护栏指标需持续监控。
- 当前结果是描述性对比，不是新旧算法因果实验结论。
- 后续进入时间、用户和内容维度拆解。
